# Daily Practice — 2026-10-04 — Statistics: Is This Really Drift, or Are You Chasing Noise?

**Dataset:** [Palmer Penguins](https://github.com/mwaskom/seaborn-data) (`penguins.csv`) — 344 real measurements of
three penguin species across three Antarctic islands (bill length/depth, flipper length, body mass), collected by
the Palmer Station Long Term Ecological Research program and distributed via `seaborn`'s built-in dataset repo.
Real field data, not synthetic.

**The scenario:** A model was trained and validated using penguins measured on **Biscoe Island** — call this the
*reference* distribution the model expects to see in production. The field team has now expanded monitoring to
**Dream** and **Torgersen** islands, and those measurements are starting to flow into the same scoring pipeline as
live *production* data. Before anyone trusts the model's output on this new stream, QA wants a drift-detection
test suite — and, because you have an SQA background, you know a detector is worthless until you've checked it
against a **known negative** (data you're certain has *not* drifted) as well as a known positive.

**What you should produce:**
1. A per-feature statistical drift test (`ks_drift_test`) using the two-sample Kolmogorov–Smirnov test, with raw
   p-values and a drift flag at a chosen significance level.
2. A **Bonferroni correction** (`bonferroni_correct`) applied across the four features tested simultaneously —
   and a short written answer to *why* testing four features at `alpha=0.05` each is not the same as a 5% false
   alarm rate overall.
3. A **Population Stability Index** (`population_stability_index`) per feature — the industry-standard
   complement to a p-value, because p-values shrink with sample size but say nothing about the *size* of a shift.
4. A **null-control check**: run your own drift test on two random halves of the reference data (which, by
   construction, have NOT drifted from each other) and confirm it doesn't cry wolf.
5. A final drift report (one row per feature: KS stat, raw p, Bonferroni-corrected flag, PSI, severity bucket)
   with a 2-3 sentence interpretation of whether this model should be flagged for retraining review.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

PENGUINS_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
NUMERIC_COLS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

raw = pd.read_csv(PENGUINS_URL)
raw.head()

## 1. Build the reference and production sets

TODO: implement `load_reference_and_production(df)`. It should:
- drop rows with any missing value in `NUMERIC_COLS`
- return `(reference_df, production_df)` where `reference_df` is every row with `island == "Biscoe"`
  and `production_df` is every row with `island` in `{"Dream", "Torgersen"}`
- reset the index on both

In [ ]:
def load_reference_and_production(df: pd.DataFrame):
    # TODO
    raise NotImplementedError

## 2. Per-feature KS drift test

TODO: implement `ks_drift_test(reference, production, columns, alpha=0.05)`. For each column name in `columns`,
run `scipy.stats.ks_2samp(reference[col], production[col])` and record the KS statistic, the p-value, and
whether `p_value < alpha`. Return a `DataFrame` indexed by feature name with columns
`ks_stat`, `p_value`, `drifted_raw`.

In [ ]:
def ks_drift_test(reference: pd.DataFrame, production: pd.DataFrame, columns: list, alpha: float = 0.05) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

## 3. Correct for multiple comparisons

TODO: implement `bonferroni_correct(pvalues, alpha=0.05)`. Given an array-like of p-values (one per feature
tested), return `(corrected_alpha, flags)` where `corrected_alpha = alpha / m` (m = number of tests) and `flags`
is a boolean array of `pvalues < corrected_alpha`.

Then, in a short markdown note of your own, answer: if you test 4 independent features at `alpha=0.05` each and
none of them have actually drifted, what is the probability that **at least one** flags as a false positive
anyway? (Hint: `1 - (1 - alpha) ** m`.)

In [ ]:
def bonferroni_correct(pvalues, alpha: float = 0.05):
    # TODO
    raise NotImplementedError

## 4. Population Stability Index (PSI)

TODO: implement `population_stability_index(reference, production, column, bins=10)`. Standard formula:
1. Define bin edges from the **reference** column's quantiles at `np.linspace(0, 1, bins + 1)` (so each
   reference bin holds roughly equal mass), with the first/last edge widened to `-inf`/`inf`.
2. Compute the fraction of reference rows and production rows falling in each bin (`ref_pct`, `prod_pct`).
3. Clip both to a small epsilon (e.g. `1e-6`) to avoid `log(0)`.
4. Return `sum((prod_pct - ref_pct) * log(prod_pct / ref_pct))`.

Industry rule of thumb: PSI < 0.1 -> no meaningful shift, 0.1-0.25 -> moderate shift worth watching,
> 0.25 -> major shift, investigate now.

In [ ]:
def population_stability_index(reference: pd.DataFrame, production: pd.DataFrame, column: str, bins: int = 10) -> float:
    # TODO
    raise NotImplementedError

## 5. Null-control check — does your detector cry wolf?

TODO: implement `null_control_check(reference, columns, alpha=0.05, seed=0)`. It should:
- randomly permute the reference rows (`np.random.RandomState(seed)`) and split them into two equal halves
  `ref_a`, `ref_b`
- run `ks_drift_test(ref_a, ref_b, columns, alpha)` and return the resulting report

If your detector is well-behaved, running it on two halves of the *same* distribution should produce **no**
flagged features (or very occasionally one, by chance — which is exactly the false-positive rate `alpha` warns
you about).

In [ ]:
def null_control_check(reference: pd.DataFrame, columns: list, alpha: float = 0.05, seed: int = 0) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

## 6. Put it together: the full drift report

TODO, using the functions above:
1. Load reference/production with `load_reference_and_production(raw)`.
2. Run `null_control_check` first — confirm no false alarms before you trust the detector on real production data.
3. Run `ks_drift_test` on reference vs. production for `NUMERIC_COLS`.
4. Apply `bonferroni_correct` to the resulting p-values.
5. Compute `population_stability_index` for each column and bucket it into "none" (<0.1), "moderate" (0.1-0.25),
   or "major" (>0.25).
6. Assemble one report `DataFrame` with columns: `ks_stat`, `p_value`, `drifted_bonferroni`, `psi`,
   `psi_severity`. Print it.
7. In a markdown cell, write 2-3 sentences: should this model be flagged for a retraining review? Which metric
   (p-value or PSI) gave you more useful information about the *size* of the problem, and why?

In [ ]:
# TODO: put it all together here

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

PENGUINS_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
NUMERIC_COLS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]


def load_reference_and_production(df: pd.DataFrame):
    clean = df.dropna(subset=NUMERIC_COLS).reset_index(drop=True)
    reference = clean[clean["island"] == "Biscoe"].reset_index(drop=True)
    production = clean[clean["island"].isin(["Dream", "Torgersen"])].reset_index(drop=True)
    return reference, production


def ks_drift_test(reference: pd.DataFrame, production: pd.DataFrame, columns: list, alpha: float = 0.05) -> pd.DataFrame:
    rows = []
    for c in columns:
        stat, p = ks_2samp(reference[c], production[c])
        rows.append({"feature": c, "ks_stat": stat, "p_value": p, "drifted_raw": p < alpha})
    return pd.DataFrame(rows).set_index("feature")


def bonferroni_correct(pvalues, alpha: float = 0.05):
    pvalues = np.asarray(pvalues)
    m = len(pvalues)
    corrected_alpha = alpha / m
    flags = pvalues < corrected_alpha
    return corrected_alpha, flags


def population_stability_index(reference: pd.DataFrame, production: pd.DataFrame, column: str, bins: int = 10) -> float:
    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.quantile(reference[column], quantiles))
    edges[0], edges[-1] = -np.inf, np.inf
    ref_counts, _ = np.histogram(reference[column], bins=edges)
    prod_counts, _ = np.histogram(production[column], bins=edges)
    ref_pct = np.clip(ref_counts / ref_counts.sum(), 1e-6, None)
    prod_pct = np.clip(prod_counts / prod_counts.sum(), 1e-6, None)
    return float(np.sum((prod_pct - ref_pct) * np.log(prod_pct / ref_pct)))


def null_control_check(reference: pd.DataFrame, columns: list, alpha: float = 0.05, seed: int = 0) -> pd.DataFrame:
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(reference))
    half = len(idx) // 2
    ref_a = reference.iloc[idx[:half]].reset_index(drop=True)
    ref_b = reference.iloc[idx[half:]].reset_index(drop=True)
    return ks_drift_test(ref_a, ref_b, columns, alpha)


raw = pd.read_csv(PENGUINS_URL)
reference, production = load_reference_and_production(raw)

null_report = null_control_check(reference, NUMERIC_COLS)
print("null control (two random halves of reference — should show no drift):")
print(null_report)

drift_report = ks_drift_test(reference, production, NUMERIC_COLS)
corrected_alpha, flags = bonferroni_correct(drift_report["p_value"].values)
drift_report["drifted_bonferroni"] = flags

psi_values = {c: population_stability_index(reference, production, c) for c in NUMERIC_COLS}
drift_report["psi"] = drift_report.index.map(psi_values)
drift_report["psi_severity"] = pd.cut(
    drift_report["psi"], bins=[-np.inf, 0.1, 0.25, np.inf], labels=["none", "moderate", "major"]
)

print(f"\nBonferroni-corrected alpha for {len(NUMERIC_COLS)} tests: {corrected_alpha:.4f}")
print(drift_report)
```

**What comes out:** the null-control check (two random halves of the *same* Biscoe population) comes back with
every p-value well above 0.05 — exactly the "no drift, correctly detected" result you want from a sanity check,
and your confidence in the real test rests on that having come back clean. The real reference-vs-production
comparison is the opposite story: all four features have p-values on the order of 1e-8 to 1e-39, every single
one survives the much stricter Bonferroni-corrected threshold (`0.05 / 4 = 0.0125`), and PSI is enormous
(roughly 0.6 for bill length, 5-7 for the other three) — "major shift" by the `>0.25` rule of thumb, several
times over.

**Why it's this extreme:** Biscoe is dominated by Gentoo penguins, a visibly larger-bodied species, while Dream
and Torgersen are almost entirely Adelie and Chinstrap. The "production" stream isn't a subtle sensor drift —
it's a near-total population replacement, so both the p-values and PSI are maxed out rather than sitting in an
ambiguous middle ground. That's a useful contrast with the null control: a well-built detector should give you
a clean "nothing to see here" on data you know hasn't moved, and an unambiguous, can't-miss-it alarm on data
that truly has. The p-value tells you the shift is statistically real (and, because KS p-values shrink with
sample size, a p-value alone can't tell you if a shift is *big* or just *measured precisely*); PSI is what
actually answers "how big," which is why production monitoring systems report both rather than either alone —
the same reason an SQA report pairs "this failed" with a severity, not just a pass/fail bit. The honest verdict:
flag this model for retraining review immediately — it was never validated on the population it's now scoring.

</details>